In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import cv2

In [2]:
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
eye_cascade   = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_eye.xml")

In [3]:
def get_croped_img(img_path):
    img = cv2.imread(img_path)
    get_gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(get_gray_img, 1.3, 5)
    for (x,y,w,h) in faces:
        roi_gray = get_gray_img[y:y+h, x:x+w]
        roi_color = img[y:y+h, x:x+w]
        eyes = eye_cascade.detectMultiScale(roi_gray)
        if len(eyes) >= 2:
            return roi_color

In [4]:
img = "C:/Users/user/Desktop/projects/classificationss/classificationss/testing/test.jpeg"
roi_img = get_croped_img(img)
# plt.imshow(roi_img)

In [6]:
import os
path_to_dir = "classification"
path_to_cr_dic = "classification/cropped"
img_dirs = []
for entry in os.scandir(path_to_dir):
    if entry.is_dir():
        img_dirs.append(entry.path)
print(img_dirs)

['classification\\not_user', 'classification\\user']


In [8]:
path_to_dir = "classification"
path_to_cr_dic = "classification/cropped"
croped_folder_dis = []
celebrity_file_name = {}
for img_dr in img_dirs:
    count = 1
    celebrity_name = img_dr.split('\\')[-1]
    celebrity_file_name[celebrity_name] = []
    for entry in os.scandir(img_dr):
        roi_color = get_croped_img(entry.path)
        if roi_color is not None:
            croped_folder = path_to_cr_dic + celebrity_name
            if not os.path.exists(croped_folder):
                os.makedirs(croped_folder)
                croped_folder_dis.append(croped_folder)
            croped_img_name = celebrity_name + str(count) + ".png"
            croped_file_path = croped_folder + "/" + croped_img_name

            cv2.imwrite(croped_file_path, roi_color)
            celebrity_file_name[celebrity_name].append(croped_file_path)
            count += 1

In [9]:
class_dis = {}
count = 0
for celebs in celebrity_file_name.keys():
    class_dis[celebs] = count
    count = count + 1
print(class_dis)


{'not_user': 0, 'user': 1}


In [13]:
import pywt
def w2d(img, mode='haar', level=1):
    imArray = img
#convert to grayscale
    imArray = cv2.cvtColor( imArray,cv2.COLOR_RGB2GRAY )
#convert to floot
    imArray = np.float32(imArray)
    imArray /= 255;
# compute coefficients
    coeffs=pywt.wavedec2(imArray, mode, level=level)
#Process Coefficients
    coeffs_H=list(coeffs)
    coeffs_H[0] *= 0;
# reconstruction
    imArray_H=pywt.waverec2(coeffs_H, mode);
    imArray_H *= 255;
    imArray_H = np.uint8(imArray_H)
    return imArray_H

In [14]:
X, y = [], []
for celebrity_name, training_files in celebrity_file_name.items():
    for training_image in training_files:
        img = cv2.imread(training_image)
        if img is None:
            continue
        scalled_raw_img = cv2.resize(img, (32, 32))
        img_har = w2d(img,'db1',5)
        scalled_img_har = cv2.resize(img_har, (32, 32))
        combined_img = np.vstack((scalled_raw_img.reshape(32*32*3,1),scalled_img_har.reshape(32*32,1)))
        X.append(combined_img)
        y.append(class_dis[celebrity_name])


In [15]:
len(X[0])

4096

In [16]:
X = np.array(X).reshape(len(X), 4096).astype(float)

In [17]:
X.shape

(32, 4096)

In [18]:
from sklearn import svm
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=10)
pipe = Pipeline([('scalar', StandardScaler()), ('svc', SVC(kernel='rbf', C=10))])
pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)

0.875

In [26]:
# len(X_test)

In [27]:
print(classification_report(y_test, pipe.predict(X_test)))

              precision    recall  f1-score   support

           0       1.00      0.75      0.86         4
           1       0.80      1.00      0.89         4

    accuracy                           0.88         8
   macro avg       0.90      0.88      0.87         8
weighted avg       0.90      0.88      0.87         8



In [28]:
import joblib
joblib.dump(pipe, "classificationmodel.h5")


['classificationmodel.h5']